In [0]:
# Common Utility Functions
# RetailMart Lakehouse

import time
import uuid
from datetime import datetime
import builtins
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
# General Helper Function

In [0]:
# Generate Run ID

def generate_run_id():
    """
    Generate a unique identifier for every pipeline execution.
    Returns = str (UUID string)
    """

    return str(uuid.uuid4())

In [0]:
# Start Pipeline

def start_pipeline():
    start_time = datetime.now()
    print(f"Pipeline Started : {start_time}")
    return start_time

In [0]:
# End Pipeline

def end_pipeline():
    end_time = datetime.now()
    print(f"Pipeline Completed : {end_time}")
    return end_time

In [0]:
# Execution Time

def execution_time(start_time, end_time):
    return builtins.round(
        (end_time-start_time).total_seconds(),2)

In [0]:
# Logging Function

from datetime import datetime

def log_message(message, level="INFO"):
    """
    Print formatted log messages with timestamp and level.
    """

    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(f"[{current_time}] [{level}] {message}")

In [0]:
# Display banner

def display_banner(notebook_name):
    """
    Displays a standard banner for every notebook.
    """
    print("RetailMart Lakehouse")
    print(f"Notebook : {notebook_name}")

In [0]:
# Profiling Utilities 

In [0]:
def dataset_profile(df, dataset_name):
    
    print(dataset_name)

    row_count = df.count()
    column_count = len(df.columns)

    print(f"Rows    : {row_count}")
    print(f"Columns : {column_count}")

    df.printSchema()

    display(df.limit(10))

In [0]:
# Execution Summary

In [0]:
#def execution_summary(notebook, rows_read, rows_written, execution_time, status):
#    summary = {
#        "Notebook": notebook,
#        "Rows Read": rows_read,
#        "Rows Written": rows_written,
#        "Duration": execution_time,
#        "Status": status
#    }
#    print("EXECUTION SUMMARY")
#    for key, value in summary.items():
#        print(f"{key:<15}: {value}")
#    return summary

In [0]:
# Validation Utilities 

In [0]:
def validate_schema(df, expected_schema):
    """
    Compares actual DataFrame schema against expected schema dict.
    expected_schema: dict like {"col_name": "type_string"}
    Returns: "PASSED" or "FAILED"
    """

    if isinstance(expected_schema, StructType):
        expected_schema = {
            field.name: field.dataType.simpleString()
            for field in expected_schema.fields
        }

    actual_schema = {
        field.name: field.dataType.simpleString()
        for field in df.schema.fields
    }

    if expected_schema == actual_schema:
        print("Schema Validation Passed")
        return "PASSED"

    print("Schema Validation Failed")
    missing = set(expected_schema) - set(actual_schema)
    extra = set(actual_schema) - set(expected_schema)
    mismatches = {
        c: (expected_schema[c], actual_schema[c])
        for c in expected_schema
        if c in actual_schema and expected_schema[c] != actual_schema[c]
    }
    if missing:
        print(f"  Missing columns : {missing}")
    if extra:
        print(f"  Extra columns   : {extra}")
    if mismatches:
        print(f"  Type mismatches : {mismatches}")
    return "FAILED"

In [0]:
# from pyspark.sql.types import StructType

# def validate_schema(df, expected_schema):

#    if isinstance(expected_schema, StructType):
#        expected_schema = {
#            field.name: field.dataType.simpleString()
#            for field in expected_schema.fields
#        }

#    actual_schema = {
#        field.name: field.dataType.simpleString()
#        for field in df.schema.fields
#    }

#    if expected_schema == actual_schema:
#        print("Schema Validation Passed")
#        return "PASSED"
#
#    print("Schema Validation Failed")
#    ...

In [0]:
def validate_primary_key(df, columns):
    """
    columns: string (single column) or list (composite key)
    Returns: True/False
    """
    if isinstance(columns, str):
        columns = [columns]

    total = df.count()
    distinct = df.select(*columns).distinct().count()
    key_label = ", ".join(columns)

    print(f"Total Rows : {total}")
    print(f"Distinct Count : {distinct}")

    if distinct == total:
        print(f"Primary Key Validation Passed — ({key_label})")
        return True

    print(f"Duplicate Key Found — ({key_label}) | Distinct: {distinct}, Total: {total}")
    return False

In [0]:
def duplicate_summary(df, total_rows):
    duplicate_count = total_rows - df.dropDuplicates().count()
    print(f"Duplicate Rows : {duplicate_count}")
    return duplicate_count

In [0]:
def null_summary(df):
    result = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])
    display(result)

In [0]:
# Validation summary

In [0]:
# Audit Utilities 

In [0]:
def add_audit_columns(df, pipeline_name, run_id):
    return (
        df
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("ingestion_date", F.current_date())
        .withColumn("pipeline_name", F.lit(pipeline_name))
        .withColumn("run_id", F.lit(run_id))
    )

In [0]:
# Delta Utilities

In [0]:
def write_bronze_table(df, target_table):
    """
    Writes DataFrame to Delta table. Returns status string ("SUCCESS"/"FAILED") and error message.
    """
    try:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
        print(f"Bronze table written: {target_table}")
        return "SUCCESS", None
    except Exception as e:
        print(f"ERROR: {str(e)}")
        return "FAILED", str(e)

In [0]:
# Report Utilities

In [0]:
def bronze_load_report(pipeline_name, run_id, source, target,
                        rows_read, rows_written, duplicate_count,
                        start_time, status,error=None):
    end_time = datetime.now()
    duration = builtins.round((end_time - start_time).total_seconds(), 2)

    print("LOAD REPORT")
    print(f"Pipeline        : {pipeline_name}")
    print(f"Run ID          : {run_id}")
    print(f"Source          : {source}")
    print(f"Target          : {target}")
    print(f"Rows Read       : {rows_read}")
    print(f"Rows Written    : {rows_written}")
    print(f"Duplicate Rows  : {duplicate_count}")
    print(f"Start Time      : {start_time}")
    print(f"End Time        : {end_time}")
    print(f"Duration (sec)  : {duration}")
    print(f"Status          : {status}")
    if error:
        print(f"Error           : {error}")